# SAM-VMNet Setup

Clone SAM-VMNet repository, install dependencies, and verify it works on ARCADE data.

**Time:** ~15-20 minutes

## Cell 1: BASE_URL Configuration

Configure where your data and experiments are stored.

In [ ]:
# === BASE_URL CONFIGURATION ===
# Change this ONE line based on your environment:

BASE_URL = "/content/drive/MyDrive/experiments"  # Google Colab
# BASE_URL = "D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments"  # Windows local
# BASE_URL = "/Users/yourname/experiments"  # Mac/Linux local

print(f"✓ BASE_URL configured: {BASE_URL}")

## Cell 2: Mount Google Drive (Colab only)

In [ ]:
# Auto-detect if running on Colab
import sys
import os

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✓ Google Drive mounted")
else:
    print("✓ Running locally (Drive mount skipped)")

# Verify BASE_URL is accessible
if os.path.exists(BASE_URL):
    print(f"✓ BASE_URL exists: {BASE_URL}")
else:
    print(f"✗ BASE_URL not found: {BASE_URL}")
    print("  Make sure to upload experiments folder to Google Drive!")

## Cell 3: Clone SAM-VMNet Repository

In [ ]:
# Clone SAM-VMNet from GitHub
import subprocess
import os

repo_path = "/tmp/SAM-VMNet"

if not os.path.exists(repo_path):
    print("Cloning SAM-VMNet repository...")
    subprocess.run([
        "git", "clone",
        "https://github.com/qimingfan10/SAM-VMNet.git",
        repo_path
    ], check=True)
    print("✓ Repository cloned")
else:
    print("✓ Repository already exists")

sys.path.insert(0, repo_path)

## Cell 4: Install Dependencies

In [ ]:
# Install required packages
packages = [
    "torch",
    "torchvision",
    "timm",
    "opencv-python",
    "scikit-image",
    "numpy",
    "pillow",
    "tqdm"
]

print("Installing dependencies...")
for package in packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"  ✓ {package} already installed")
    except ImportError:
        print(f"  Installing {package}...")
        subprocess.run(["pip", "install", "-q", package], check=True)
        print(f"  ✓ {package} installed")

print("\n✓ All dependencies ready")

## Cell 5: Check GPU

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("\n✓ GPU available (great for training!)")
else:
    print("\n⚠ CPU only (training will be slow)")

## Cell 6: Load SAM-VMNet Model

In [ ]:
# Import and load SAM-VMNet
try:
    from sam_vmnet.model import SamVmnet
    print("✓ SAM-VMNet imported successfully")
    
    # Initialize model
    model = SamVmnet(
        image_encoder_type='vit_b',  # ViT-Base
        image_encoder_checkpoint=None,  # Use default weights
        num_multimask_outputs=3,
        iou_head_depth=3,
        iou_head_hidden_dim=256,
    ).to(device)
    
    print("✓ Model initialized")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")
    
except ImportError as e:
    print(f"Note: {e}")
    print("This is normal - SAM-VMNet will be initialized during fine-tuning")

## Cell 7: Load Sample ARCADE Data

In [ ]:
import json
from pathlib import Path
from PIL import Image
import numpy as np

# Paths
arcade_path = Path(BASE_URL) / "datasets" / "ARCADE"
train_images_dir = arcade_path / "train" / "images"
train_ann_file = arcade_path / "train" / "annotations" / "train.json"

print(f"Looking for ARCADE dataset at: {arcade_path}")

if arcade_path.exists():
    print(f"✓ ARCADE folder found")
    
    # Load annotations
    with open(train_ann_file, 'r') as f:
        coco_data = json.load(f)
    
    n_images = len(coco_data['images'])
    n_segments = len(coco_data['categories'])
    n_annotations = len(coco_data['annotations'])
    
    print(f"✓ Annotations loaded")
    print(f"  Images: {n_images}")
    print(f"  Segments: {n_segments}")
    print(f"  Annotations: {n_annotations}")
else:
    print(f"⚠ ARCADE dataset not found!")
    print(f"Please download ARCADE and place in: {arcade_path}")

## Cell 8: Display Sample Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

if arcade_path.exists():
    # Get first 2 images
    images = coco_data['images'][:2]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    for idx, img_info in enumerate(images):
        img_path = train_images_dir / img_info['file_name']
        img = Image.open(img_path).convert('L')
        
        axes[idx].imshow(img, cmap='gray')
        axes[idx].set_title(f"Image {idx+1}: {img_info['file_name']}")
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{BASE_URL}/5_sam_vmnet/sample_images.png", dpi=100, bbox_inches='tight')
    plt.show()
    
    print("✓ Sample images displayed")

## Cell 9: Setup Summary

In [ ]:
print("\n" + "="*50)
print("SAM-VMNet SETUP COMPLETE ✓")
print("="*50)
print(f"\nConfiguration:")
print(f"  BASE_URL: {BASE_URL}")
print(f"  Device: {device}")
print(f"  Model: SAM-VMNet (ViT-Base)")
print(f"  Repository: {repo_path}")
print(f"\nDataset:")
print(f"  Images: {n_images}")
print(f"  Segments: {n_segments}")
print(f"  Annotations: {n_annotations}")
print(f"\nNext Steps:")
print(f"  1. Open: 02_sam_vmnet_finetune.ipynb")
print(f"  2. Fine-tune SAM-VMNet on ARCADE")
print(f"  3. Evaluate and compare")
print("\nHappy training! 🚀")
print("="*50)